Project Overview:
1. Data Preprocessing

Filter active users and popular movies to reduce sparsity

Map original userId and movieId to matrix indices

Create user-item rating matrix (sparse format for ALS)

2. Collaborative Filtering (ALS)

Use Alternating Least Squares (ALS) from implicit library

Train model on user-item matrix

Obtain top N recommendations for each user

3. Content-Based Features

Clean tags and merge with genres

Combine genres + tags into a single text field per movie

Use TF-IDF vectorization to represent movies in feature space

4. Hybrid Recommendation

Compute cosine similarity of ALS top N movies with all movies

Combine ALS scores + content similarity scores

Rank movies to produce final hybrid recommendations

## Ratings:
| Column      | Meaning                   |
| ----------- | ------------------------- |
| `userId`    | Unique ID for each user   |
| `movieId`   | Unique ID for each movie  |
| `rating`    | User’s rating (1–5 stars) |
| `timestamp` | When the rating was given |


In [19]:
import pandas as pd
import numpy as np

In [20]:
ratings = pd.read_csv(r'C:\Users\connect\Downloads\ml-32m\ml-32m\ratings.csv' , nrows = 100_000)

In [21]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [22]:
ratings.shape

(100000, 4)

In [23]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100000 non-null  int64  
 1   movieId    100000 non-null  int64  
 2   rating     100000 non-null  float64
 3   timestamp  100000 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [24]:
ratings.duplicated().sum()

np.int64(0)

In [25]:
ratings.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [26]:
# how many ratings each user has made
user_counts = ratings['userId'].value_counts()
# basic statistics about user rating counts
user_counts.describe()

count     626.000000
mean      159.744409
std       251.187220
min         6.000000
25%        39.000000
50%        76.500000
75%       164.000000
max      2842.000000
Name: count, dtype: float64

In [27]:
# Users with very few ratings must be filtered out to reduce noise and improve the quality 
# of recommendations. This helps in focusing on more active users who provide better insights
min_user_ratings = 20

In [28]:
# Which movies are rated enough times to be considered popular
movie_counts = ratings['movieId'].value_counts()
# basic statistics about movie rating counts
movie_counts.describe()

count    10225.000000
mean         9.779951
std         21.840793
min          1.000000
25%          1.000000
50%          2.000000
75%          8.000000
max        319.000000
Name: count, dtype: float64

In [29]:
# Most movies have very few ratings, Such items must be filtered to improve recommendation quality and reduce sparsity
min_movie_ratings = 20

## Filtered_Ratings 

In [30]:
# Filter active users
active_users = user_counts[user_counts >= min_user_ratings].index

# Filter popular movies
popular_movies = movie_counts[movie_counts >= min_movie_ratings].index

# Apply both filters
filtered_ratings = ratings[
    ratings['userId'].isin(active_users) &
    ratings['movieId'].isin(popular_movies)
]

filtered_ratings.shape

(67131, 4)

## Collaborative Filtering

Prepare data for ALS:

In [31]:
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

import numpy as np

user_ids = filtered_ratings['userId'].unique()
movie_ids = filtered_ratings['movieId'].unique()

user_to_index = {u: i for i, u in enumerate(user_ids)}
movie_to_index = {m: i for i, m in enumerate(movie_ids)}

index_to_movie = np.array(movie_ids)   #THIS replaces LabelEncoder


filtered_ratings['user_idx'] = filtered_ratings['userId'].map(user_to_index)
filtered_ratings['movie_idx'] = filtered_ratings['movieId'].map(movie_to_index)

#Build the Sparse User_Item Matrix
from scipy.sparse import csr_matrix

user_item = csr_matrix(
    (
        filtered_ratings['rating'],
        (filtered_ratings['user_idx'], filtered_ratings['movie_idx'])
    ),
    shape=(len(user_ids), len(movie_ids))
)


print(user_item.shape)  


(625, 1306)


C:\Users\connect\AppData\Local\Temp\ipykernel_9776\3277892907.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_ratings['user_idx'] = filtered_ratings['userId'].map(user_to_index)
C:\Users\connect\AppData\Local\Temp\ipykernel_9776\3277892907.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_ratings['movie_idx'] = filtered_ratings['movieId'].map(movie_to_index)


In [32]:
# number of non-zero values
user_item.nnz

67131

In [33]:
import numpy as np
np.isnan(user_item.data).any()

np.False_

ALS-based Collaborative Filtering

In [34]:
# pip install implicit

In [35]:
ratings_cf = filtered_ratings.copy()

In [36]:
from implicit.als import AlternatingLeastSquares

model = AlternatingLeastSquares(
    factors=50,
    regularization=0.01,
    iterations=20,
    random_state=42
)

# implicit expects ITEM × USER
model.fit(user_item.T)


c:\Users\connect\miniconda3\envs\ml_env\lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.003974437713623047 seconds
  warnings.warn(


  0%|          | 0/20 [00:00<?, ?it/s]

In [37]:
# original userId from CSV
userId = 7

# Map to filtered matrix row
user_idx = user_to_index[userId]
items, scores = model.recommend(
    userid = user_idx,
    user_items=user_item[user_idx],
    N=5
)

items, scores

(array([ 64, 480, 529,  82, 532], dtype=int32),
 array([1.3562315, 1.310615 , 1.3047278, 1.3016427, 1.2374299],
       dtype=float32))

In [38]:
items.max(), user_item.shape[1]

(np.int32(532), 1306)

## Movies

In [39]:
movies = pd.read_csv(r'C:\Users\connect\Downloads\ml-32m\ml-32m\movies.csv')
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [40]:
movies.shape

(87585, 3)

In [41]:
movies.duplicated().sum()

np.int64(0)

In [42]:
movies.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [43]:
movies.describe(include='O')

,title,genres
count,87585,87585
unique,87382,1798
top,Alone (2020),Drama
freq,4,12443


## Use Movies to recommend the titles and genres

In [44]:
original_movie_ids = index_to_movie[items]
original_movie_ids

array([ 1653,  6383, 45499,  2324, 45720])

In [45]:
recommended_movies = movies[movies['movieId'].isin(original_movie_ids)]
recommended_movies[['movieId', 'title' , 'genres']]

,movieId,title,genres
1591,1653,Gattaca (1997),Drama|Sci-Fi|Thriller
2233,2324,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama|Romance|War
6265,6383,"2 Fast 2 Furious (Fast and the Furious 2, The)...",Action|Crime|Thriller
10806,45499,X-Men: The Last Stand (2006),Action|Sci-Fi|Thriller
10837,45720,"Devil Wears Prada, The (2006)",Comedy|Drama


## Tags 

In [46]:
tags = pd.read_csv(r'C:\Users\connect\Downloads\ml-32m\ml-32m\tags.csv')  
tags.head()

,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102


In [47]:
# Keep those columns
tags = tags[['movieId', 'tag']]

In [48]:
tags.duplicated().sum()

np.int64(913231)

In [49]:
tags = tags.drop_duplicates()

In [50]:
tags.duplicated().sum()

np.int64(0)

In [51]:
tags['tag'] = tags['tag'].str.lower()
tags.head()

,movieId,tag
0,26479,kevin kline
1,79592,misogyny
2,247150,acrophobia
3,2174,music
4,2174,weird


## Hybrid Recommender System combines Collaborative Filtering (CF) + Content-Based (CB)

In [52]:
# Merge tags with movies safely
movies_content = movies.copy()

In [53]:
# Make sure tags are strings, fill NaN with empty string
tags['tag'] = tags['tag'].fillna('').astype(str).str.lower()

In [54]:
# Group by movieId and join all tags
tags_per_movie = tags.groupby('movieId')['tag'].apply(lambda x: " ".join(x)).reset_index()

In [55]:
# Merge with movies
movies_content = movies_content.merge(tags_per_movie, on='movieId', how='left')

In [56]:
# Fill any missing tags
movies_content['tag'] = movies_content['tag'].fillna('')

In [57]:
# Combine genres + tags
movies_content['content'] = movies_content['genres'].str.replace('|', ' ', regex=False) + ' ' + movies_content['tag']
movies_content[['movieId', 'title', 'content']].head()

,movieId,title,content
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy ch...
1,2,Jumanji (1995),Adventure Children Fantasy robin williams fant...
2,3,Grumpier Old Men (1995),Comedy Romance comedinha de velhinhos engraãƒâ...
3,4,Waiting to Exhale (1995),Comedy Drama Romance characters slurs based on...
4,5,Father of the Bride Part II (1995),Comedy fantasy pregnancy remake family steve m...


In [58]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies_content['content']) # Fit and transform the content

print(tfidf_matrix.shape)  # num_movies, num_features


(87585, 46657)


In [59]:
import numpy as np

userId = 7
user_idx = user_to_index[userId]

N = 5  # top N ALS recommendations
als_items, als_scores = model.recommend(user_idx, user_item[user_idx], N=N)


In [60]:
# Build movieId / tfidf index mapping
movie_idx_map = {mid: i for i, mid in enumerate(movies_content['movieId'])}
als_indices = [movie_idx_map[mid] for mid in index_to_movie[als_items]]

In [61]:
from sklearn.metrics.pairwise import cosine_similarity

# cosine similarity between ALS top N movies and all movies
similarity_scores = cosine_similarity(tfidf_matrix[als_indices], tfidf_matrix)

# sum similarity for each movie
content_scores = similarity_scores.sum(axis=0)


In [62]:
# scores
combined_scores = np.zeros(len(movies_content))

In [63]:
# Assign ALS scores to their movies
for i, idx in enumerate(als_indices):
    combined_scores[idx] = als_scores[i]

combined_scores += 0.5 * content_scores  # weight content by 0.5

In [64]:
# Get top 5 hybrid recommendations
top_idx = combined_scores.argsort()[::-1][:5]
hybrid_recommendations = movies_content.iloc[top_idx][['movieId', 'title', 'genres']]
hybrid_recommendations

,movieId,title,genres
1591,1653,Gattaca (1997),Drama|Sci-Fi|Thriller
2233,2324,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama|Romance|War
10806,45499,X-Men: The Last Stand (2006),Action|Sci-Fi|Thriller
6265,6383,"2 Fast 2 Furious (Fast and the Furious 2, The)...",Action|Crime|Thriller
10837,45720,"Devil Wears Prada, The (2006)",Comedy|Drama


## Save 

In [65]:
import joblib
from pathlib import Path

# base metadata path
METADATA_PATH = Path(r"C:\Users\connect\Downloads\Recommendation_Project\app\metadata")

# Save ALS model
joblib.dump(model, METADATA_PATH / "als_model.pkl")

# Save user-item matrix
joblib.dump(user_item, METADATA_PATH / "user_item_matrix.pkl")

# Save movies content (DataFrame)
movies_content.to_pickle(METADATA_PATH / "movies_content.pkl")

# Save TF-IDF matrix
joblib.dump(tfidf_matrix, METADATA_PATH / "tfidf_matrix.pkl")

# Save mapping dictionaries
joblib.dump(user_to_index, METADATA_PATH / "user_to_index.pkl")
joblib.dump(index_to_movie, METADATA_PATH / "index_to_movie.pkl")


['C:\\Users\\connect\\Downloads\\Recommendation_Project\\app\\metadata\\index_to_movie.pkl']

In [66]:
# Save movies_content
movies_content.to_parquet("C:\\Users\\connect\\Downloads\\Recommendation_Project\\Data\\movies_content.parquet", index=False)

# Save filtered ratings
ratings_cf.to_parquet("C:\\Users\\connect\\Downloads\\Recommendation_Project\\Data\\ratings_cf.parquet", index=False)

# Save tags 
tags.to_parquet("C:\\Users\\connect\\Downloads\\Recommendation_Project\\Data\\tags.parquet", index=False)